# EDA

In [ ]:
import json
import pandas as pd

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

In [2]:
with open("../data/skills_2511.json", encoding="utf-8") as f:
    tax = pd.DataFrame(json.load(f))

tax.head()

,skill_id,skill,headlines
0,13ad_dCC_Tn6,"OOS, styr- och utvecklingsmodell",[Styr- och utvecklingsmodeller]
1,16ZN_3Zp_bYV,"Odoo, molnplattform",[Applikationsplattformar]
2,19PK_uxU_Wzy,"Bower, pakethanterare",[Programmerings- och systemutvecklingsverktyg]
3,1CmG_8hR_igc,Bluetooth-teknik,[Kommunikationsprotokoll]
4,1QYQ_Xka_TQ5,"Microsoft Autopilot, systemutvecklingsverktyg",[Programmerings- och systemutvecklingsverktyg]


## Jobbannonser EDA

### Datasetets grundläggande struktur

In [3]:
# Snabb överblick
print(f"Datasetet innehåller {df.shape[0]} rader (annonser) och {df.shape[1]} kolumner.\n")
print("---------------------------")
print(df.info())
print("---------------------------")

print("\n Dataset kolumner utdrag:")
print(df.head())
print("---------------------------")

print("\nFeatures i datasetet:\n", df.columns)
print("---------------------------")

Datasetet innehåller 6894 rader (annonser) och 10 kolumner.

---------------------------
<class 'pandas.DataFrame'>
RangeIndex: 6894 entries, 0 to 6893
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            6894 non-null   str  
 1   version_id    6894 non-null   str  
 2   title         6894 non-null   str  
 3   employer      6894 non-null   str  
 4   municipality  5935 non-null   str  
 5   region        5935 non-null   str  
 6   occupation    6894 non-null   str  
 7   published     6894 non-null   str  
 8   deadline      6894 non-null   str  
 9   description   6894 non-null   str  
dtypes: str(10)
memory usage: 538.7 KB
None
---------------------------

 Dataset kolumner utdrag:
         id                                version_id  \
0  29354136  abbb7874b4c0d5792c74dcfec2be2d1a825e9579   
1  29354152  1f17787e47533f40dbf0df610e26a4f8b9d790e7   
2  29354331  d127a2a996c37ab43c33c5607ad285fde7cbdc50

### Datakvalitet

In [4]:
# Saknade värden + tomma rader
missing = (
    df.isna() |
    df.astype(str).apply(lambda col: col.str.strip().eq(""))
).sum()

print(missing)

print("---------------------------")

missing_percentage = (
    missing / len(df) * 100
).sort_values(ascending=False)

print(missing_percentage)
print("---------------------------")


id                0
version_id        0
title             0
employer          0
municipality    959
region          959
occupation        0
published         0
deadline          0
description       8
dtype: int64
---------------------------
municipality    13.910647
region          13.910647
description      0.116043
id               0.000000
version_id       0.000000
employer         0.000000
title            0.000000
occupation       0.000000
published        0.000000
deadline         0.000000
dtype: float64
---------------------------


In [5]:
#Dubletter
print("Unika annons-ID:", df["id"].nunique())
print("Totalt antal annonser:", len(df))
print("Antal duplicerade rader:", df.duplicated().sum())


# dubletter baserade på tid + arbetsgivare + beskrivning
duplicate_ads = df.duplicated(
    subset=["title", "employer", "description"]
).sum()

print("Potentiella duplicerade annonser:", duplicate_ads) 
print("---------------------------")

duplicate_texts = df[df["description"].duplicated(keep=False)]
duplicate_texts[["id","title","employer","published","description"]].head(20)
print("Potentiella duplicerade annonser:", duplicate_texts.sum)
print("---------------------------")

Unika annons-ID: 6637
Totalt antal annonser: 6894
Antal duplicerade rader: 0
Potentiella duplicerade annonser: 947
---------------------------
Potentiella duplicerade annonser: <bound method DataFrame.sum of             id                                version_id  \
6     29355065  80fee065cdecb056026d75c18c095d96f64d2ec6   
8     29355229  6bd58c9893a5bc98dc7adee09f49739847c31409   
9     29355231  014909a0d4c27fdcb093b4d3767b584c8e9432f7   
10    29355214  c5c61da24ef32514b2eb5fdf58c01ffd85ba944d   
11    29355216  8db1ce02ea99e1259580ff54ac55d76eb1379c6d   
...        ...                                       ...   
6810  31387962  c8651c346f21a0ae402c3d58debd9aafeaa1ed6f   
6831  31395395  05504137f5ed0b7309491b1c29f9cef9630b0ab9   
6854  31406416  95095d07abc5aac8eca11e4100fb84a073c30623   
6885  31417585  99902288970c762523e2041e68933935f7441aa3   
6893  31420250  07d419ac0f3ad5432d611652160640c2249b2da9   

                                                  title  \
6     Consul

#### Fördelning av jobbannonser

In [6]:
#Titlar
unique_titles = df["title"].nunique()

print(f"Antal annonser: {len(df)}")
print(f"Antal unika jobbtitlar: {unique_titles}")
print(f"Genomsnittligt antal annonser per titel: {len(df) / unique_titles:.2f}")
print("---------------------------")

print(df["title"].value_counts().head(20))
print("---------------------------")

Antal annonser: 6894
Antal unika jobbtitlar: 4549
Genomsnittligt antal annonser per titel: 1.52
---------------------------
title
Lösningsarkitekt                              95
Kravanalytiker                                89
Solution Architect                            62
Data Scientist                                48
Jönköping: Senior IT-arkitekt hybridtjänst    48
IT-arkitekt                                   43
Business Analyst                              33
Enterprise Architect                          30
Lösningsarkitekt till Softhouse i Luleå       28
Senior Data Scientist                         25
Senior Kravanalytiker                         24
Informationsarkitekt                          24
Enterprisearkitekt                            22
Configuration Manager                         21
Data Analyst                                  19
Senior Solution Architect                     18
Sales Representative                          18
IT-konsult                           

In [7]:
# Yrkesgrupper enligt SSYK (occupation)
unique_occupations = df["occupation"].nunique()

print(f"Antal unika yrkesgrupper: {unique_occupations}")
print("---------------------------")

print(f"Genomsnittligt antal annonser per yrke: {len(df) / unique_occupations:.2f}")
print("---------------------------")

print(df["occupation"].value_counts().head(20))
print("---------------------------")

Antal unika yrkesgrupper: 16
---------------------------
Genomsnittligt antal annonser per yrke: 430.88
---------------------------
occupation
IT-arkitekt/Lösningsarkitekt       2072
Affärskonsult, IT                  1058
Systemanalytiker/Systemutredare     861
Kravanalytiker                      629
Systemarkitekt                      489
Verksamhetskonsult, IT              368
Enterprise Architect                296
Infrastrukturarkitekt               292
IT-arkitekt                         271
Data scientist                      244
Configuration Manager/CM            119
IT-utredare                          93
Metodanalytiker, IT                  70
Informationsarkitekt                 19
Informatiker                          7
Nätverksarkitekt                      6
Name: count, dtype: int64
---------------------------


In [8]:
# Geografisk fördelning
# enligt kommuner
print("Antal unika kommuner:", df["municipality"].nunique())
print("---------------------------")
print(df["municipality"].value_counts().head(20))
print("---------------------------")

# enligt region
print("Antal unika regioner:", df["region"].nunique())
print("---------------------------")
print(df["region"].value_counts().head(20))
print("---------------------------")

Antal unika kommuner: 119
---------------------------
municipality
Stockholm      2330
Göteborg        714
Malmö           296
Solna           271
Linköping       169
Jönköping       168
Sundbyberg      160
Luleå           153
Lund            136
Södertälje      108
Norrköping       93
Örebro           83
Sundsvall        81
Västerås         73
Växjö            61
Helsingborg      56
Enköping         55
Uppsala          55
Karlstad         50
Umeå             39
Name: count, dtype: int64
---------------------------
Antal unika regioner: 21
---------------------------
region
Stockholms län          2964
Västra Götalands län     880
Skåne län                511
Östergötlands län        307
Jönköpings län           184
Norrbottens län          169
Uppsala län              117
Örebro län               105
Västernorrlands län      102
Kronobergs län            95
Västmanlands län          92
Västerbottens län         58
Värmlands län             51
Dalarnas län              50
Kalmar län   

In [9]:
# datumfördelning
df["published"] = pd.to_datetime(df["published"])

df["year"] = df["published"].dt.year
df["month"] = df["published"].dt.month

# Annonser per år
print(df["year"].value_counts().sort_index())
print("---------------------------")

# Annonser per månad
print(df["month"].value_counts().sort_index())
print("---------------------------")

# Annonser per år + månad
df["year_month"] = df["published"].dt.to_period("M")
monthly_ads = df["year_month"].value_counts().sort_index()

print(monthly_ads)
print("---------------------------")

# Vilken månad + år hade mest annonser
print(monthly_ads.idxmax())
print(monthly_ads.max())
print("---------------------------")


year
2025    4425
2026    2469
Name: count, dtype: int64
---------------------------
month
1      786
2      835
3      744
4     1012
5      779
6      589
7      411
8      442
9      350
10     333
11     260
12     353
Name: count, dtype: int64
---------------------------
year_month
2025-01    479
2025-02    457
2025-03    420
2025-04    619
2025-05    466
2025-06    285
2025-07    225
2025-08    178
2025-09    350
2025-10    333
2025-11    260
2025-12    353
2026-01    307
2026-02    378
2026-03    324
2026-04    393
2026-05    313
2026-06    304
2026-07    186
2026-08    264
Freq: M, Name: count, dtype: int64
---------------------------
2025-04
619
---------------------------


In [10]:
#Tabell med annonser per månad per år
year_month_table = pd.crosstab(
    df["month"],
    df["year"]
)

print(year_month_table)

year   2025  2026
month            
1       479   307
2       457   378
3       420   324
4       619   393
5       466   313
6       285   304
7       225   186
8       178   264
9       350     0
10      333     0
11      260     0
12      353     0


In [11]:
# datumintervallet
print("Första annonsen:", df["published"].min())
print("Senaste annonsen:", df["published"].max())

Första annonsen: 2025-01-02 08:28:27
Senaste annonsen: 2026-08-31 20:29:35


In [12]:
# beskrivningarna
# längd
df["description_length"] = df["description"].str.len()
print(df["description_length"].describe())
print("---------------------------")

# antal tecken
df["word_count"] = (
    df["description"]
    .str.split()
    .str.len()
)
print(df["word_count"].describe())
print("---------------------------")

# tomma descriptions
print(
    "Tomma descriptions:",
    df["description"].isna().sum()
)
print(
    "Descriptions < 100 tecken:",
    (df["description_length"] < 100).sum()
)

count    6894.000000
mean     3151.942559
std      1462.435350
min         0.000000
25%      2082.000000
50%      3073.000000
75%      4237.750000
max      6506.000000
Name: description_length, dtype: float64
---------------------------
count    6894.000000
mean      449.558167
std       207.696712
min         0.000000
25%       294.000000
50%       441.000000
75%       610.000000
max       978.000000
Name: word_count, dtype: float64
---------------------------
Tomma descriptions: 0
Descriptions < 100 tecken: 12


## Taxonomi EDA

### Taxonomi grundläggande struktur

In [13]:
# Snabb överblick
print(f"Taxonomi innehåller {tax.shape[0]} rader (annonser) och {tax.shape[1]} kolumner.\n")
print("---------------------------")
print(tax.info())
print("---------------------------")

print("\n Taxonmi kolumner utdrag:")
print(tax.head())
print("---------------------------")

print("\nFeatures i taxonomi:\n", tax.columns)
print("---------------------------")

Taxonomi innehåller 204 rader (annonser) och 3 kolumner.

---------------------------
<class 'pandas.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   skill_id   204 non-null    str   
 1   skill      204 non-null    str   
 2   headlines  204 non-null    object
dtypes: object(1), str(2)
memory usage: 4.9+ KB
None
---------------------------

 Taxonmi kolumner utdrag:
       skill_id                                          skill  \
0  13ad_dCC_Tn6               OOS, styr- och utvecklingsmodell   
1  16ZN_3Zp_bYV                            Odoo, molnplattform   
2  19PK_uxU_Wzy                          Bower, pakethanterare   
3  1CmG_8hR_igc                               Bluetooth-teknik   
4  1QYQ_Xka_TQ5  Microsoft Autopilot, systemutvecklingsverktyg   

                                        headlines  
0                 [Styr- och utvecklingsmodeller]  
1                

### Datakvalitet

In [14]:
# Saknade värden
missing = tax.isna().sum()

print(missing)
print("---------------------------")

missing_percentage = (
    tax.isna().mean() * 100
).sort_values(ascending=False)

print(missing_percentage)
print("---------------------------")

#Dubletter
print("Unika skill-ID:", tax["skill_id"].nunique())
print("Totalt antal annonser:", len(tax))

skill_id     0
skill        0
headlines    0
dtype: int64
---------------------------
skill_id     0.0
skill        0.0
headlines    0.0
dtype: float64
---------------------------
Unika skill-ID: 204
Totalt antal annonser: 204


In [15]:
skill_headline = tax[["skill", "headlines"]].copy()

print(skill_headline.to_string(index=False))

                                                             skill                                      headlines
                                  OOS, styr- och utvecklingsmodell                [Styr- och utvecklingsmodeller]
                                               Odoo, molnplattform                      [Applikationsplattformar]
                                             Bower, pakethanterare [Programmerings- och systemutvecklingsverktyg]
                                                  Bluetooth-teknik                      [Kommunikationsprotokoll]
                     Microsoft Autopilot, systemutvecklingsverktyg [Programmerings- och systemutvecklingsverktyg]
            Office SharePoint Server (MOSS), applikationsplattform                      [Applikationsplattformar]
                                  Ledningssystem (ISO 9000-serien)                              [Kvalitetssystem]
                                                        WAP-teknik                      